In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
data=pd.read_csv('all_tickets_processed.csv')

In [ ]:
data.head()

,Document,Topic_group,Ticket Priority
0,connection with icon icon dear please setup ic...,Hardware,Medium
1,work experience user work experience user hi w...,Access,Medium
2,requesting for meeting requesting meeting hi p...,Hardware,Medium
3,reset passwords for external accounts re expir...,Access,Medium
4,mail verification warning hi has got attached ...,Miscellaneous,Medium


In [ ]:
data.loc[0,'Document']

'connection with icon icon dear please setup icon per icon engineers please let other details needed thanks lead'

In [ ]:
data['Topic_group'].value_counts()

,count
Topic_group,
Hardware,13617
HR Support,10915
Access,7125
Miscellaneous,7060
Storage,2777
Purchase,2464
Internal Project,2119
Administrative rights,1760


In [ ]:
data['Ticket Priority'].value_counts()

,count
Ticket Priority,
Medium,20071
Low,11260
High,10652
Critical,5854


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,LabelEncoder,OrdinalEncoder
def make_preprocessor():
    return ColumnTransformer(transformers=[
        ('vect', TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True
        ), 'Document')
    ])
priority_order = [["Low", "Medium", "High", "Critical"]]
oe = OrdinalEncoder(categories=priority_order)
ohe=OneHotEncoder()
le=LabelEncoder()



In [ ]:
y=pd.DataFrame({'Ticket Priority':oe.fit_transform(data[['Ticket Priority']]).ravel(),
               "Topic_group"    : le.fit_transform(data["Topic_group"])})
X=data.drop(columns=['Ticket Priority','Topic_group'])

In [ ]:
X.head()

,Document
0,connection with icon icon dear please setup ic...
1,work experience user work experience user hi w...
2,requesting for meeting requesting meeting hi p...
3,reset passwords for external accounts re expir...
4,mail verification warning hi has got attached ...


In [ ]:
y.head()

,Ticket Priority,Topic_group
0,1.0,3
1,1.0,0
2,1.0,3
3,1.0,0
4,1.0,5


In [ ]:
X.head()

,Document
0,connection with icon icon dear please setup ic...
1,work experience user work experience user hi w...
2,requesting for meeting requesting meeting hi p...
3,reset passwords for external accounts re expir...
4,mail verification warning hi has got attached ...


In [ ]:
y_priority=y['Ticket Priority']
y_grp=y['Topic_group']

In [ ]:
from sklearn.model_selection import train_test_split

X_grp_train,X_grp_test,y_grp_train,y_grp_test=train_test_split(X,y_grp,random_state=42,test_size=0.2,stratify=y["Topic_group"])
X_pri_train,X_pri_test,y_pri_train,y_pri_test=train_test_split(X,y_priority,random_state=42,test_size=0.2,stratify=y["Ticket Priority"])


In [ ]:
y_grp_train.value_counts()

,count
Topic_group,
3,10893
2,8732
0,5700
5,5648
7,2222
6,1971
4,1695
1,1408


In [ ]:
y_pri_train.value_counts()

,count
Ticket Priority,
1.0,16057
0.0,9008
2.0,8521
3.0,4683


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline

In [ ]:
pipe_lr=Pipeline([
    ('preprocessor',make_preprocessor()),
    ('classifier',LogisticRegression())
])

pipe_rf=Pipeline([
    ('preprocessor',make_preprocessor()),
    ('classifier',RandomForestClassifier( class_weight="balanced",
        n_estimators=200,
        random_state=42,
        n_jobs=-1))
])

**MODEL FOR PREDICTION OF TOPIC GROUP(LOGISTIC REGRESSTION)**

In [ ]:
pipe_lr.fit(X_grp_train,y_grp_train)
y_grp_pre=pipe_lr.predict(X_grp_test)
print("REPORT OF LOGISTIC REGRESSION\n",classification_report(y_grp_test,y_grp_pre))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


REPORT OF LOGISTIC REGRESSION
               precision    recall  f1-score   support

           0       0.91      0.88      0.89      1425
           1       0.88      0.67      0.76       352
           2       0.87      0.88      0.88      2183
           3       0.81      0.90      0.85      2724
           4       0.91      0.81      0.86       424
           5       0.83      0.83      0.83      1412
           6       0.98      0.86      0.91       493
           7       0.94      0.82      0.88       555

    accuracy                           0.86      9568
   macro avg       0.89      0.83      0.86      9568
weighted avg       0.86      0.86      0.86      9568



In [ ]:
pipe_rf.fit(X_pri_train,y_pri_train)
y_pri_pre=pipe_rf.predict(X_pri_test)
print("REPORT OF RANDOM FOREST\n",classification_report(y_pri_test,y_pri_pre))


REPORT OF RANDOM FOREST
               precision    recall  f1-score   support

         0.0       0.88      0.73      0.80      2252
         1.0       0.77      0.91      0.84      4014
         2.0       0.79      0.77      0.78      2131
         3.0       1.00      0.78      0.87      1171

    accuracy                           0.82      9568
   macro avg       0.86      0.80      0.82      9568
weighted avg       0.83      0.82      0.82      9568



In [ ]:
def predict(ticket_text):

    input_df = pd.DataFrame([{"Document": ticket_text}])

    grp_num  = pipe_lr.predict(input_df)[0]
    print(pipe_lr.predict(input_df))

    print(grp_num)

    grp_word = le.inverse_transform([int(grp_num)])[0]
    print(le.inverse_transform([int(grp_num)]))
    print(grp_word)
    pri_num  = pipe_rf.predict(input_df)[0]

    pri_word = oe.inverse_transform([[pri_num]])[0][0]
    print(oe.inverse_transform([[pri_num]]))

    print(f"Input    : {ticket_text}")
    print(f"Type     : {grp_word}")
    print(f"Priority : {pri_word}")
    print()

# ── Test ──────────────────────────────────────────────
predict("My laptop screen is broken and I cannot work")
predict("Please reset my password I am locked out")
predict("Need access to the finance module for new intern")
predict("Running low on mailbox storage please increase limit")

[3]
3
['Hardware']
Hardware
[['High']]
Input    : My laptop screen is broken and I cannot work
Type     : Hardware
Priority : High

[0]
0
['Access']
Access
[['High']]
Input    : Please reset my password I am locked out
Type     : Access
Priority : High

[2]
2
['HR Support']
HR Support
[['Low']]
Input    : Need access to the finance module for new intern
Type     : HR Support
Priority : Low

[7]
7
['Storage']
Storage
[['Medium']]
Input    : Running low on mailbox storage please increase limit
Type     : Storage
Priority : Medium

